# EcoSort Waste Management Assistant
# Module 8 Summative Lab

## Overview

You are a data scientist at "EcoSort," a technology company that specializes in developing AI solutions for waste management. EcoSort has partnered with Metro City's waste management department to develop an intelligent waste management assistant that can help residents properly dispose of waste items so less time is spent sorting material at facilities.

This assistant needs to:

1. Identify waste materials from images uploaded by residents (CNN)
2. Classify waste items based on text descriptions provided by residents (RNN/Transformer)
3. Generate specific recycling instructions based on identified waste type and city policies (Generative Transformer with RAG)

Your task is to build this integrated system using the RealWaste dataset along with generated text data that simulates real-world waste management operations.

# Part 3: Waste Description Classification

In this section, you will build a text classification model to categorize waste based on descriptions.

### 3.1 Preprocess Text Data
This subsection confirms that the cleaned, stratified, and tokenized text datasets are ready for modelling.


# Code Overview: Text preprocessing validation
Displays dataset splits, category coverage, cleaned text samples, and tokenized outputs to verify successful preprocessing, stratification, and tokenization before model training.

In [ ]:

# Applying the text preprocessing pipeline

print("Training records:", len(text_train_df))
print("Validation records:", len(text_validation_df))
print("Test records:", len(text_test_df))

print("\nNumber of Categories:")
print(len(class_names))

print("\nCategories:")
for category in class_names:
    print("-", category)

# Display a few cleaned examples
display(
    text_df[
        [
            "description",
            "clean_description",
            "category"
        ]
    ].head()
)

# Verify tokenized dataset shapes
print("\nTokenized Dataset Shapes")

print(
    "Training:",
    train_text_encoding["input_ids"].shape
)

print(
    "Validation:",
    validation_text_encoding["input_ids"].shape
)

print(
    "Test:",
    test_text_encoding["input_ids"].shape
)

### Text preprocessing validation summary
The text preprocessing pipeline successfully preserved all nine categories and produced stratified training (4,000), validation (500), and test (500) records, ensuring consistent category representation and reliable model evaluation across all datasets.

### 3.2 Implement Text Classification Model
TF-IDF captures informative words and phrases, while Logistic Regression provides a fast and interpretable multiclass classifier for the short descriptions.

## Code Overview: TF-IDF and Logistic Regression Implementation
Transforms waste descriptions into TF-IDF unigram and bigram features to create numerical representations for effective multiclass classification using Logistic Regression.

In [ ]:

# Implement TF-IDF + Logistic regression model for text classification

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Creating TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words="english"
)

# Transform text into TF-IDF features
X_train = tfidf_vectorizer.fit_transform(
    text_train_df["clean_description"]
)

X_validation = tfidf_vectorizer.transform(
    text_validation_df["clean_description"]
)

X_test = tfidf_vectorizer.transform(
    text_test_df["clean_description"]
)

# Target variables
y_train = text_train_df["label"]
y_validation = text_validation_df["label"]
y_test = text_test_df["label"]

print("TF-IDF feature matrices created successfully.")

print(f"Training shape: {X_train.shape}")
print(f"Validation shape: {X_validation.shape}")
print(f"Test shape: {X_test.shape}")

### Code Overview: Logistic regression model development
Builds a reproducible multiclass Logistic Regression classifier to learn waste category patterns from TF-IDF text features.

### 3.3 Train and Evaluate the Model

In [ ]:
# Build the logistic regression model


text_model = LogisticRegression(
    max_iter=2000,
    random_state=SEED
)

print("Logistic Regression model created successfully.")

## Justification for model selection
TF-IDF with Logistic Regression was chosen over transformer-based approaches because the short waste descriptions are efficiently represented by sparse lexical features, delivering strong performance, high interpretability, fast inference, and low computational cost, while DistilBERT was retained as an exploratory benchmark.

## Train and Evaluate the Model
The classifier is trained and assessed on unseen descriptions using accuracy, class-level metrics, and error analysis.

## Code Overview: Logistic regression model training
Trains the Logistic Regression classifier on TF-IDF features and evaluates validation accuracy to learn waste-category patterns and assess generalization performance.

In [ ]:
#Training the logistic regression model

text_model.fit(
    X_train,
    y_train
)

print("Text classification model trained successfully.")

# Training accuracy
train_accuracy = text_model.score(
    X_train,
    y_train
)

# Validation accuracy
validation_accuracy = text_model.score(
    X_validation,
    y_validation
)

print(f"\nTraining Accuracy: {train_accuracy:.4f}")
print(f"Validation Accuracy: {validation_accuracy:.4f}")

## Code Brief: Text model inference and evaluation
Generates waste category predictions and confidence scores to evaluate model performance and support classification of new text inputs.

In [ ]:
# Evaluating the model performance

# Generating predictions on the test dataset
y_pred = text_model.predict(X_test)

# Calculate test accuracy
test_accuracy_text = accuracy_score(
    y_test,
    y_pred
)

print(f"Test Accuracy: {test_accuracy_text:.4f}")


## Code Overview: Text classification report
Generates precision, recall, and F1-scores for each waste category to provide a detailed assessment of text classification performance beyond overall accuracy.



In [ ]:
# CLASSIFICATION REPORT


report = classification_report(
    y_test,
    y_pred,
    target_names=class_names
)

print(report)

### Code Overview: Text confusion matrix analysis
Visualizes actual versus predicted waste categories to identify classification accuracy and common sources of text classification errors.

In [ ]:
# Confusion matrix

cm_text = confusion_matrix(
    y_test,
    y_pred
)

plt.figure(figsize=(10, 8))

sns.heatmap(
    cm_text,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.title("Text Classification Confusion Matrix")
plt.xlabel("Predicted Category")
plt.ylabel("Actual Category")

plt.tight_layout()
plt.show()

## Code Overview: Misclassification analysis
Analyzes incorrectly classified descriptions to identify common confusion patterns, ambiguous inputs, and opportunities for improving text classification performance.

In [ ]:
# Analyzing misclassifications

# Creating a comparison dataset
error_analysis_df = text_test_df.copy()

error_analysis_df["Predicted_Label"] = y_pred

error_analysis_df["Predicted_Category"] = (
    text_label_encoder.inverse_transform(y_pred)
)

error_analysis_df["Correct"] = (
    error_analysis_df["category"]
    == error_analysis_df["Predicted_Category"]
)

# Extract misclassified records
misclassified = error_analysis_df[
    error_analysis_df["Correct"] == False
]

print(
    f"Misclassified Records: {len(misclassified)}"
)

display(
    misclassified[
        [
            "description",
            "category",
            "Predicted_Category"
        ]
    ].head(20)
)
misclassified.head(10)

### Text classification error analysis
Most errors occurred between categories with similar wording patterns, such as Textile Trash and Miscellaneous Trash or Plastic and mixed packaging materials. With only 2 misclassifications out of 500 test samples, the model demonstrates excellent generalization and class discrimination.

### Code Overview: Challenging category identification
Ranks waste categories by classification accuracy to identify the most difficult classes and support targeted analysis of recurring confusion patterns and model weaknesses.

In [ ]:
#Identifying challenging categories

category_accuracy = (
    cm_text.diagonal()
    / cm_text.sum(axis=1)
)

category_performance = pd.DataFrame({
    "Category": class_names,
    "Accuracy": category_accuracy
})

category_performance = (
    category_performance
    .sort_values(
        "Accuracy",
        ascending=True
    )
)

display(category_performance)

print("\nMost Challenging Categories")

display(
    category_performance.head(3)
)

### Text classification performance summary
The text classifier achieved 99.6% test accuracy, with only 2 errors across 500 unseen descriptions, demonstrating excellent generalization. Although Textile Trash and Miscellaneous Trash were the lowest-performing categories at 98.28% accuracy, performance remained consistently strong across all waste classes.

### 3.4 Create Classification Function
This subsection packages the text pipeline and classifier into a reusable function for the integrated assistant.

### Code Overview: Text classification function
Creates a reusable inference function that preprocesses user descriptions, applies TF-IDF transformation, and predicts the corresponding waste category.

In [ ]:
# TODO: Create a function that takes a text description and returns the predicted waste category

def classify_waste_description(description):
    """
    Classifies a waste description into an appropriate category.

    Args:
        description (str): Text description of waste item

    Returns:
        str: Predicted waste category
    """
    # Clean the incoming text
    cleaned_text = (
        str(description)
        .lower()
        .strip()
    )

    # Convert text into TF-IDF features
    text_features = tfidf_vectorizer.transform(
        [cleaned_text]
    )

    # Generate prediction
    predicted_label = text_model.predict(
        text_features
    )[0]

    # Convert the numeric label back to the category name
    predicted_category = (
        text_label_encoder
        .inverse_transform([predicted_label])[0]
    )

    return predicted_category

### Code Overview: Text classification function validation
Tests the text classification function on sample descriptions to verify accurate waste-category prediction and reliable inference on new user inputs.

In [ ]:
#Test the clasification function

sample_descriptions = [
    "empty plastic water bottle",
    "used cardboard shipping box",
    "broken glass bottle",
    "banana peel and vegetable waste",
    "old cotton shirt"
]

for description in sample_descriptions:

    prediction = classify_waste_description(
        description
    )

    print(
        f"Description: {description}"
    )

    print(
        f"Predicted Category: {prediction}\n"
    )

### Text classification function validation summary
The classification function correctly identified all five sample descriptions, demonstrating consistent preprocessing, accurate category prediction, and readiness for integration into the waste classification system.